In [1]:
from pathlib import Path
import cv2

from configuration.config import yolo_raw_dir


# ==========================
# Configuration
# ==========================
IMAGE_DIR = yolo_raw_dir / "images"
LABEL_DIR = yolo_raw_dir / "labels"
OUTPUT_DIR = yolo_raw_dir / "cropped"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_EXTS = [".jpg", ".jpeg", ".png"]


def yolo_to_xyxy(label, img_w, img_h):
    cls, xc, yc, w, h = map(float, label.split())

    xc *= img_w
    yc *= img_h
    w *= img_w
    h *= img_h

    x1 = int(max(0, xc - w / 2))
    y1 = int(max(0, yc - h / 2))
    x2 = int(min(img_w, xc + w / 2))
    y2 = int(min(img_h, yc + h / 2))

    return int(cls), x1, y1, x2, y2


count = 0

for img_path in IMAGE_DIR.iterdir():

    if img_path.suffix.lower() not in IMG_EXTS:
        continue

    label_path = LABEL_DIR / (img_path.stem + ".txt")

    if not label_path.exists():
        print(f"Missing label: {img_path.name}")
        continue

    img = cv2.imread(str(img_path))

    if img is None:
        print(f"Cannot read {img_path}")
        continue

    h, w = img.shape[:2]

    with open(label_path, "r") as f:
        labels = [line.strip() for line in f if line.strip()]

    for idx, label in enumerate(labels):

        cls, x1, y1, x2, y2 = yolo_to_xyxy(label, w, h)

        crop = img[y1:y2, x1:x2]

        if crop.size == 0:
            continue

        save_name = f"{img_path.stem}_{idx:03d}.jpg"

        cv2.imwrite(str(OUTPUT_DIR / save_name), crop)

        count += 1

print(f"Finished. Saved {count} cropped images.")

Finished. Saved 1416 cropped images.
